# 🇨🇺 Generador de Diccionario Inverso (Cubanismos -> Español Estándar)
Este notebook está diseñado para correr en Google Colab con una GPU T4 gratuita.
Utiliza un modelo ligero (ej. Llama-3-8B-Instruct o Qwen-1.5) para generar las traducciones de cada lema en la base de datos.

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas sqlite3

### 1. Sube tu base de datos
Ejecuta la siguiente celda y sube el archivo `diccionario_cubanismos.db` (en el que ya corrimos la migración) desde tu computadora.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
db_filename = list(uploaded.keys())[0]
print(f"Archivo {db_filename} cargado con éxito!")

### 2. Cargar el Modelo LLM (Qwen 1.5 4B Chat - Ligero y excelente en Español)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Puedes cambiarlo por tu modelo de 'cecilIA' si lo tienes subido a HuggingFace
model_id = "Qwen/Qwen1.5-4B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
print("Modelo cargado.")

### 3. Función de Traducción

In [ ]:
def traducir_cubanismo(lema):
    prompt = f"""<|im_start|>system
Eres un lingüista experto en dialectología hispánica. Tu tarea es traducir palabras del dialecto cubano al español neutro/estándar internacional. Responde ÚNICAMENTE con el sinónimo o frase corta equivalente, sin explicaciones.<|im_end|>
<|im_start|>user
Traduce el cubanismo: '{lema}'<|im_end|>
<|im_start|>assistant
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=10, 
        temperature=0.3, 
        do_sample=True, 
        eos_token_id=tokenizer.eos_token_id
    )
    
    # Extraer la respuesta generada
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip().title()

### 4. Procesar la Base de Datos y Descargar

In [ ]:
import sqlite3
import pandas as pd
from tqdm.auto import tqdm

conn = sqlite3.connect(db_filename)
cursor = conn.cursor()

# Obtener lemas sin traducción
cursor.execute("SELECT rowid, lema FROM cubanismos WHERE traduccion IS NULL OR traduccion = ''")
filas = cursor.fetchall()

print(f"Se encontraron {len(filas)} cubanismos sin traducir.")

# Procesar uno por uno
for rowid, lema in tqdm(filas):
    try:
        traduccion = traducir_cubanismo(lema)
        cursor.execute("UPDATE cubanismos SET traduccion = ? WHERE rowid = ?", (traduccion, rowid))
        # Guardar cada 10 registros
        if rowid % 10 == 0: conn.commit()
    except Exception as e:
        print(f"Error al procesar '{lema}': {e}")

conn.commit()
conn.close()

print("¡Traducción completada! Preparando descarga...")
files.download(db_filename)